In [1]:
# %matplotlib widget
import os
import sys
import numpy as np
import matplotlib
# matplotlib.rcParams['font.family'] = 'Times New Roman'
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LogNorm
from mpl_toolkits import mplot3d
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
import math
from IPython.display import HTML
from tqdm import tqdm
import random
from numpy.lib import recfunctions as rfn
import multiprocessing

from sklearn.cluster import DBSCAN
from scipy.interpolate import interp1d
from scipy.spatial import KDTree
from scipy.interpolate import interpn
# import cupy as cp
import time
import numba as nb

# 画图部分
import matplotlib as mpl
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import ImageGrid
from functools import partial
import matplotlib.cm as cm

# ROC曲线部分
from sklearn.metrics import roc_curve, auc
from sklearn import metrics

# 插值
import torch
import torch.cuda as cuda
from scipy.interpolate import griddata

# 系数拟合部分
from scipy.optimize import curve_fit
from sklearn import svm


# 概率比较部分
from scipy.special import gammaln
import torch.distributions as dist

# 位置重建函数
import sys
sys.path.append('..')
import position_rec as pos

import DE
from DE import DESimulation
import importlib

# from thundersvm import SVC

Trans into torch tensor input


In [2]:
cevns_color = '#5882F8'
DE_color = '#F9885E'

In [3]:
importlib.reload(DE)
importlib.reload(plt)
from DE import DESimulation

辅助函数
=====

In [4]:
def rndm(a, b, g, size=1):
    """Power-law gen for pdf(x)\propto x^{g-1} for a<=x<=b"""
    r = np.random.random(size=size)
    ag, bg = a**g, b**g
    return (ag + (bg - ag)*r)**(1./g)

def dense_muon_gen(modified_merged_muon_array, interp_num, event_time, dead_time_ratio):
    '''生成dense_muon_points数据, 为保证速度，从muon track的首尾线性插值，并且能量均匀分布'''

    # 定义每个元素的类型
    element_types = [('eventId', '<i4'), ('energy', '<f8'), ('xd', '<f8'), ('yd', '<f8'),
                     ('zd', '<f8'), ('muon_time', '<f8'), ('num_e', '<u4'), ('num_e_delayed', '<u4')]
    # 定义muon模拟数量
    muon_sim_num = (modified_merged_muon_array['eventId'][-1] + 1)
    dense_muon_points = np.zeros(
        int(muon_sim_num*interp_num), dtype=element_types)
    # 获取modified_merged_muon_array中每个eventId刷新坐标
    unique_elements, start_indices = np.unique(
        modified_merged_muon_array['eventId'], return_index=True)
    end_indices = np.concatenate(
        [start_indices[1:] - 1, np.array([len(modified_merged_muon_array)-1,])])

    start_point, end_point = modified_merged_muon_array[
        start_indices], modified_merged_muon_array[end_indices]
    gap_x = - start_point['xd'] + end_point['xd']
    gap_y = - start_point['yd'] + end_point['yd']
    gap_z = - start_point['zd'] + end_point['zd']

    # 创建长度为100的0-1之间差值数组
    arr = np.tile(np.arange(0, 100/99, 1/99), (muon_sim_num, 1))
    x_dense, y_dense, z_dense = arr * \
        gap_x[:, np.newaxis], arr * \
        gap_y[:, np.newaxis], arr * gap_z[:, np.newaxis]
    x_dense, y_dense, z_dense = x_dense + start_point['xd'][:, np.newaxis], y_dense + \
        start_point['yd'][:, np.newaxis], z_dense + \
        start_point['zd'][:, np.newaxis]

    dense_muon_points['xd'], dense_muon_points['yd'], dense_muon_points['zd'] = x_dense.flatten(
    ), y_dense.flatten(), z_dense.flatten()
    dense_muon_points['eventId'] = (
        np.arange(muon_sim_num*interp_num)/100).astype('uint32')

    # 计算每个muon沉积能量总和及电子总和

    # 提取 eventId 和 electron_num 列
    event_ids = modified_merged_muon_array['eventId']
    electron_nums = modified_merged_muon_array['electronNum']
    muon_step_energy = modified_merged_muon_array['energy']

    # 使用 numpy.unique 获取唯一 eventId 和对应的索引
    unique_event_ids, event_id_indices = np.unique(
        event_ids, return_inverse=True)

    # 使用 numpy.bincount 计算每个相同 eventId 的 electron_num 总和
    muon_electron_sum = np.bincount(event_id_indices, weights=electron_nums)
    muon_electron_energy = np.bincount(
        event_id_indices, weights=muon_step_energy)
    muon_electron_sum.astype('uint32')

    # 补全dense_muon_points中的energy，num_e以及num_e_delayed
    average_electron = (muon_electron_sum/100).astype('uint32')
    average_energy = (muon_electron_energy/100).astype('uint32')

    dense_muon_points['energy'] = average_energy[dense_muon_points['eventId']]
    dense_muon_points['num_e'] = average_electron[dense_muon_points['eventId']]
    dense_muon_points['muon_time'] = event_time[dense_muon_points['eventId']]

    # 定义残留比例
    delay_ratio = ((dense_muon_points['zd']-100) / (-1.7) / 1000 + 0.02) / 100
    delay_ratio[delay_ratio < 0.001] = 0.001
    dense_muon_points['num_e_delayed'] = dense_muon_points['num_e'] * \
        delay_ratio * dead_time_ratio

    return dense_muon_points

类型定义
======

In [5]:
data = np.load("../cevns_sim.npz")

# 创建一个字典来复制数据
new_data_dict = {key: data[key] for key in data.files}

# 修改特定的数据
new_data_dict['CEvNS'] = new_data_dict['CEvNS'] * 0 + 1

new_data_dict['CEvNS'][3] = 1e4

np.savez('3e.npz',**new_data_dict)

# # 修改特定的数据
# new_data_dict['CEvNS'] = new_data_dict['CEvNS'] * 0 + 1

# new_data_dict['CEvNS'][4] = 1e5

# np.savez('4e.npz',**new_data_dict)

# # 修改特定的数据
# new_data_dict['CEvNS'] = new_data_dict['CEvNS'] * 0 + 1

# new_data_dict['CEvNS'][5] = 1e5

# np.savez('5e.npz',**new_data_dict)

# # 修改特定的数据
# new_data_dict['CEvNS'] = new_data_dict['CEvNS'] * 0 + 1

# new_data_dict['CEvNS'][6] = 1e5

# np.savez('6e.npz',**new_data_dict)

In [6]:
file_load_once = 10
param = 0

cevns_sim = DESimulation('DE_param_30_3e.yaml')

# test_cevns_sim.pe_gain_std = 2.4e6

muon_file_list = []
for i in range(file_load_once):
    # print(i)
    file_index = param * file_load_once + i
    filename = cevns_sim.load_folder_path + \
        "muon_track/muon_track." + str(file_index) + ('.npy')
    # print(filename)
    muon_file_list.append(filename)

In [7]:
cevns_sim.generate_muon_points(muon_file_list)
cevns_sim.generate_CEVNS_points()
cevns_sim.generate_CEVNS_pattern()
cevns_sim.CEVNS_position_recon()
cevns_sim.generate_CEVNS_recon_pattern()
cevns_sim.CEVNS_sp_cor()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 195.67it/s]


/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.0.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.1.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.2.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.3.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.4.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.5.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.6.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.7.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.8.npy
/home/cchang/RELICS_DE_sim/file_to_load/muon_track/muon_track.9.npy
Number of muon events: 31255
merged_muon_array data type: [('eventId', '<i4'), ('type', 'i1'), ('energy', '<f8'), ('xd', '<f8'), ('yd', '<f8'), ('zd', '<f8'), ('td', '<f8')]
dense_muon_points data type:  [('eventId', '<i4'), ('energy', '<f8'), ('xd', '<f8'), ('yd', '<f8'), ('zd', '<f8'), ('muon_time', '<f8'), ('num_e',

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9/9 [00:00<00:00, 45153.99it/s]


Start pos recon for event size:  2
(1, 64)
1
done of generating test files
Start pos recon for event size:  3
(10000, 64)
10000
10000 / 10000 ,time: 0.09873127937316895
done of generating test files
Start pos recon for event size:  4
(1, 64)
1
done of generating test files
Start pos recon for event size:  5
(1, 64)
1
done of generating test files
Start pos recon for event size:  6
(1, 64)
1
done of generating test files
Start pos recon for event size:  7
(1, 64)
1
done of generating test files
Start pos recon for event size:  8
(1, 64)
1
done of generating test files
Start pos recon for event size:  9
(1, 64)
1
done of generating test files
Interpn for channel:  0
Interpn for channel:  1
Interpn for channel:  2
Interpn for channel:  3
Interpn for channel:  4
Interpn for channel:  5
Interpn for channel:  6
Interpn for channel:  7
Interpn for channel:  8
Interpn for channel:  9
Interpn for channel:  10
Interpn for channel:  11
Interpn for channel:  12
Interpn for channel:  13
Interpn for

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 2552.83it/s]


...

平均电子数：  6.974915736690368
last_muon_id.shape:  (1,)
last_muon_id[-10:]:  [4617]
0
0 1


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1948.12it/s]


平均电子数：  7.3655188195580745
last_muon_id.shape:  (1,)
last_muon_id[-10:]:  [1030]
0
0 1


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1966.39it/s]

平均电子数：  10.226674779909873


In [8]:
# pattern 绘图

test_DE_sim = DESimulation('DE_param_30_3e.yaml')
def pattern_plot(phe_pmt, electron = [] , pos_recons = [], pmt_info = test_DE_sim.pmt_info,title = "",savefig = False):
    artists = list()
    # cmap = mpl.cm.viridis
    # 定义灰度色图
    cmap = cm.get_cmap("viridis")
    norm = mpl.colors.LogNorm(vmin=1, vmax=phe_pmt.max(), clip=True)
    colors = ['blue', 'green', 'yellow','red']  # 自定义颜色列表
    #norm = mpl.colors.LogNorm(vmin=1, vmax=1000, clip=True)
    fig = plt.figure(figsize=(7, 6))
    plt.style.use('../relics.mplstyle') 

    # grid = ImageGrid(fig, 111, (1,1), cbar_mode='single', axes_pad=0.1, cbar_pad=0.2)
    grid = ImageGrid(fig, 111, (1,1))

    # PMT plot
    # for pmt_id in range(64):
    #     pmt = pmt_info[pmt_id]
    #     pe_value = phe_pmt[pmt_id]
    #     # print(str(round(pe_value,2)))
    #     # grid[0].text(pmt['x'], pmt['y'], str(pmt['ChannelID']), ha='center', va='center')
    #     grid[0].text(pmt['x'], pmt['y'], str(round(pe_value,1)), ha='center', va='center',fontsize = 15)
    #     # artists.append(Rectangle(xy=(pmt['x']-1.27, pmt['y']-1.27), width=2.54, height=2.54, angle=pmt['rot_z'], rotation_point='center', fc=cmap(norm(phe_pmt[pmt_id])),alpha = 0.86))
    #     # grid[0].add_patch(artists[pmt['ChannelID']])
    for pmt_id in range(64):
        pmt = pmt_info[pmt_id]
        pe_value = phe_pmt[pmt_id]
        # 检查 PMT 值并设定颜色
        if phe_pmt[pmt_id] < 0.5:
            pmt_color = 'white'  # 小于0.1的通道颜色为白色
        else:
            pmt_color = cmap(norm(phe_pmt[pmt_id]))
            # 绘制alpha为1的白色边框
        # grid[0].text(pmt['x'], pmt['y'], str(round(pe_value,1)), ha='center', va='center',fontsize = 15)
        border = Rectangle(xy=(pmt['x']-1.27, pmt['y']-1.27), width=2.54, height=2.54, angle=pmt['rot_z'], rotation_point='center', fc='none', ec='black', lw=1, alpha=1)    
        grid[0].add_patch(border)

        # artists.append(Rectangle(xy=(pmt['x']-1.27, pmt['y']-1.27), width=2.54, height=2.54, 
        #                angle=pmt['rot_z'], rotation_point='center', fc=pmt_color, 
        #                alpha=(phe_pmt[i][pmt_id] / vmax)**0))

        artists.append(Rectangle(xy=(pmt['x']-1.27, pmt['y']-1.27), width=2.54, height=2.54, 
                       angle=pmt['rot_z'], rotation_point='center', fc=pmt_color, 
                       alpha=1))
        grid[0].add_patch(artists[pmt['ChannelID']])
    # event plot
    if len(electron)>0:
        grid[0].plot(electron['xd']/10, electron['yd']/10, 'ro', markersize=12 ,label = 'DE Position')
  
    # recon_pos plot
    if len(pos_recons)>0:
        grid[0].plot(pos_recons['xd']/10, pos_recons['yd']/10, 'b*', markersize=12, label = 'Recon Position')
        
    # muon_point_plot
    # if muon_id_list is not None:
        # for i, muon_id in enumerate(muon_id_list):
            # color = colors[i % len(colors)]  # 从颜色列表中选择颜色
            # grid[0].plot(dense_muon_points[dense_muon_points['eventId'] == muon_id]['xd']/10, dense_muon_points[dense_muon_points['eventId'] == muon_id]['yd']/10, 'o', label="muon id: {}".format(muon_id), markersize=2, alpha=0.5, color=color)
            
    # TPC outline
    angle = np.arange(13)*np.pi/6
    outer_r = 14/np.cos(np.pi/12)
    grid[0].plot(outer_r*np.cos(angle), outer_r*np.sin(angle), c='k', lw=3)

    # Colorbar
    # grid.cbar_axes[0].colorbar(mpl.cm.ScalarMappable(cmap=cmap, norm=norm))

    # Other settings
    grid[0].axis('equal')
    grid[0].set_xlabel('x/cm')
    grid[0].set_ylabel('y/cm')
    # grid[0].legend(bbox_to_anchor=(1.2, 0), loc=3, borderaxespad=0, prop={'size': 7})
    grid[0].set_title(title)
    if savefig == True: 
        plt.savefig(title+".png")
    plt.show()
    return 0

# pattern计算
def calculate_poisson_probabilities_log(data_array, mean_array):
    data_array_int = (data_array + 0.5).astype('int')
    # 计算每个元素的对数概率
    log_probabilities = -mean_array + data_array_int * \
        np.log(mean_array) - gammaln(data_array_int + 1)
    return log_probabilities

In [9]:
cevns_sim.cevns_points[1].shape

(10000,)

In [10]:
# cevns_points = np.concatenate(cevns_sim.cevns_points)
cevns_points = cevns_sim.cevns_points[1]

# for i in range(100,103):
#     pattern_plot(cevns_points[i]['recons_light_pattern'],
#                  cevns_points[i]['pos_original'],
#                  cevns_points[i]['pos_recon'],
#                  title = 'Light Pattern')

CEVNSpe，pattern和st_correlation分析
=====

In [8]:
def calculate_poisson_probabilities_log(data_array, mean_array):
    # 计算每个元素的对数概率
    log_probabilities = -mean_array + data_array * np.log(mean_array) - gammaln(data_array + 1)
    return log_probabilities

In [9]:
# 定义cevns点的三参数
cevns_info = np.zeros(len(cevns_points),dtype = np.dtype([('area', '<f8'),('st_cor', '<f8'),('pattern', '<f8'),('st_cor_new', '<f8'),('e_num','uint32')]))
cevns_info['area'] = np.sum(cevns_points['pe_by_area'],axis = 1)
cevns_info['st_cor'] = np.log(cevns_points['st_cor'])
# cevns_info['st_cor_new'] = np.log(cevns_points['st_cor_new'])
cevns_info['pattern'] = np.sum(calculate_poisson_probabilities_log(
         cevns_points['pe_by_area'][:,:64], cevns_points['recons_light_pattern'][:,:64]),axis = 1)


In [ ]:
# 绘制二维直方图
fig = plt.figure(figsize=(8, 6), dpi=80)  # 调整 figsize 参数以设置合适的宽度和高度
#plt.hist2d(real_space_time_cor_veto, real_log_l_veto_4, bins=500 , cmap='hot', norm=LogNorm(),alpha = 0.7,label='Real Event')
#plt.colorbar(pad = -0.01)
H = plt.hist2d(cevns_info['st_cor'], cevns_info['pattern'],  bins = 500, cmap='cool', norm=LogNorm(),alpha = 0.7,label = 'DE Event')
#mask = test_DE_sim.CEVNS_sp_cor[0] > 0
# H = plt.hist2d(np.log(test_DE_sim.CEVNS_sp_cor[0][mask]), CEVNS_pattern_possion[0][mask],  bins = 500, cmap='hot', norm=LogNorm(),alpha = 0.7,label = 'CEVNS Event')
plt.xlabel('Space_time_correlation',fontsize = 14)
plt.ylabel('Pattern Log Likelihood',fontsize = 14)
plt.xlim(-10, 25)
plt.ylim(-225,-25)
plt.grid()
plt.show()

In [ ]:
x_bins = np.arange(-30,50,0.5)
y_bins = np.arange(-300,0,1)

pattern划分，按照90-240pe每10pe为一个bin进行分析
=====

In [ ]:
# area_low_lim,area_up_lim = 90,240

In [10]:
k = np.load('0523_20000_sigma.npy')
[k_st,k_area,b] = k

In [ ]:
cevns_info.dtype

In [ ]:
# area_mask = (cevns_info['area'] > area_low_lim) * (cevns_info['area'] < area_up_lim)
# cevns_info_roi = cevns_info[area_mask]
score_cevns = cevns_info['pattern'] - (k_st * cevns_info['st_cor']) + k_area * cevns_info['area'] + b

In [ ]:
# 设定分数阈值的范围
thresholds = np.arange(-400, 200, 1)  # 你可以调整这个范围和步长以适应你的数据

# 初始化两个比例的列表
right_cevns_ratios = []

# 计算每个阈值下的比例
for threshold in thresholds:
    right_cevns_ratio = np.sum(score_cevns > threshold) / len(score_cevns)
    right_cevns_ratios.append(right_cevns_ratio)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 创建一个新的图形并设置大小
fig, ax1 = plt.subplots(figsize=(10, 7),dpi = 80)

# 第一个图，使用左侧y轴
ax1.set_xlabel('Score', fontsize=14)  # x轴标签
ax1.set_ylabel('DE Cut \& CEVNS Survival Rate', fontsize=14)  # 左侧y轴标签
ax1.plot(thresholds, right_cevns_ratios, label=r"CE$\nu$NS Survival Rate", color='darkblue')  # 绘制右侧 CEVNS 的比例曲线

# 开始处理第二个图，使用右侧y轴
ax2 = ax1.twinx() # 使用两个不同的y轴

bins = np.arange(-400, 100, 1)


# 计算并绘制CEVNS得分的频率
hist_cevns_score, bin_edges = np.histogram(score_cevns, bins=bins, density=True)
ax2.plot(bin_edges[:-1], hist_cevns_score, drawstyle='steps', fillstyle='none', label=r"CE$\nu$NS", alpha=0.5, color='darkblue')

# 获得所有子图的legend labels和handles
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# 创建一个全局的legend
fig.legend(lines + lines2, labels + labels2, loc=(0.69,0.71))

fig.tight_layout(pad = 5)  # 使布局紧密
plt.title('Pattern Classifier', fontsize=16)  # 图表标题
plt.xlim(-300, 200)
plt.axvline(x=0, color='red', linestyle='dashed')
plt.show()  # 显示绘图

In [ ]:
# 导入所需要的库
import matplotlib.pyplot as plt
import numpy as np

# 创建一个新的图形并设置大小
fig, ax1 = plt.subplots()
plt.rcParams['text.usetex'] = True

# 开始处理第一个图，使用左侧y轴

color = 'tab:blue'
bins = np.arange(-400, 150, 1)
ax1.hist(score_cevns, bins=bins, log=True, label=r"CE$\nu$NS", alpha=0.7, color=cevns_color,density = True)  # 添加 alpha 参数
ax1.set_ylabel('Frequency', fontsize=14)  # 右侧y轴标签
ax1.set_xlabel('Pattern Classifier Score', fontsize=14)  # x轴标签
ax1.tick_params(axis='y')  # y轴标签颜色

# 第二个图，使用右侧y轴
ax2 = ax1.twinx() 
ax2.set_ylabel(r"Survival Rate", fontsize=14)  # 左侧y轴标签
ax2.plot(thresholds, right_cevns_ratios, label=r"CE$\nu$NS Survival Rate", color=cevns_color)  # 绘制右侧 CEVNS 的比例曲线
ax2.grid(True)
ax2.tick_params(axis='y')  # y轴标签颜色


# 获得所有子图的 legend labels 和 handles
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

# 创建一个全局的legend
# fig.legend(lines + lines2, labels + labels2, loc=(0.65,0.71))
fig.legend(lines, labels, loc=(0.70,0.71))

# fig.tight_layout(pad = 5)  # 使布局紧密
# plt.title('Pattern Classifier', fontsize=16)  # 图表标题
plt.xlim(-250, 150)
plt.axvline(x=0, color='red', linestyle='dashed')
plt.show()  # 显示绘图


In [ ]:
cevns_info_mask = score_cevns > 0
acceptance_pattern = len(score_cevns[cevns_info_mask]) / len(score_cevns)
acceptance_pattern

In [11]:
sigma_se = 190e-9

def cevns_waveform_gen(pe_info,sigma_se,z):
    
    pe_num = len(pe_info)
    
    Dl = 12
    vd = 0.174 * 1000000

    # cevns_z = np.random.uniform(0,24)
    cevns_z = z 
    t = cevns_z / vd
    # sigma_d = math.sqrt((2 * Dl * t) / (vd ** 2) + sigma_0 ** 2)
    sigma_t = np.sqrt((2 * Dl * t) / (vd ** 2) )
    sigma = sigma_t
    pe_e_time = np.zeros(len(pe_info))
    
    for e_id in np.unique(pe_info['e_id']):
        e_time = np.random.normal(0,sigma) 
        mask = pe_info['e_id'] == e_id
        pe_e_time[mask] = e_time
    # print("#########", pe_e_time)
    
    se_signal_delay = np.random.normal(0,sigma_se,pe_num)
    pe_time = pe_e_time + se_signal_delay
    pe_time = pe_time - np.mean(pe_time)
    # print(pe_time)
    sample_index = pe_time / (4e-9)
    sample_index = sample_index.astype(int)
    # print(sample_index)
    waveform = np.zeros(3500)
    np.add.at(waveform,sample_index + 1750,pe_info['area']/6e6)
    std = np.std(sample_index) * 4e-9
    # waveform[sample_index + 1750] += (pe_info['area']/6e6)
    # print(pe_info['area']/6e6)
    # fig = plt.figure(figsize=(7, 4))
    # plt.plot(waveform[1000:-1000])
    # plt.show()
    return waveform,std

def top_waveform_gen(pe_info,sigma_se):
    
    pe_num = len(pe_info)
    
    Dl = 12
    vd = 0.174 * 1000000

    cevns_z = 0
    t = cevns_z / vd
    # sigma_d = math.sqrt((2 * Dl * t) / (vd ** 2) + sigma_0 ** 2)
    sigma_t = np.sqrt((2 * Dl * t) / (vd ** 2) )
    sigma = sigma_t
    pe_e_time = np.zeros(len(pe_info))
    
    for e_id in np.unique(pe_info['e_id']):
        e_time = np.random.normal(0,sigma) 
        mask = pe_info['e_id'] == e_id
        pe_e_time[mask] = e_time
    # print("#########", pe_e_time)
    
    se_signal_delay = np.random.normal(0,sigma_se,pe_num)
    pe_time = pe_e_time + se_signal_delay
    pe_time = pe_time - np.mean(pe_time)
    # print(pe_time)
    sample_index = pe_time / (4e-9)
    sample_index = sample_index.astype(int)
    # print(sample_index)
    waveform = np.zeros(3500)
    np.add.at(waveform,sample_index + 1750,pe_info['area']/6e6)
    std = np.std(sample_index) * 4e-9
    # waveform[sample_index + 1750] += (pe_info['area']/6e6)
    # print(pe_info['area']/6e6)
    # fig = plt.figure(figsize=(7, 4))
    # plt.plot(waveform[1000:-1000])
    # plt.show()
    return waveform,std

In [12]:
e_num_list = np.arange(3,9)
remain_cevns_id_list = []
accumulate_num = 0
for i in range(2):
# for i in range(len(cevns_sim.cevns_e_spectrum)):
    
    cevns_points = cevns_sim.cevns_points[i]
    e_num = cevns_sim.cevns_e_num[i]
    # cevns_id = cevns_points['cevnsID'] - accumulate_num
    cevns_id = np.arange(0,len(cevns_points))
    # print(np.min(cevns_id))
    
    # st_mask 
    st_mask = (cevns_points['st_cor'] > 0)
    
    cevns_info = np.zeros(len(cevns_points[st_mask]),dtype = np.dtype([('area', '<f8'),('st_cor', '<f8'),('pattern', '<f8')]))
    cevns_info['area'] = np.sum(cevns_points[st_mask]['pe_by_area'],axis = 1)
    cevns_info['st_cor'] = np.log(cevns_points[st_mask]['st_cor'])
    cevns_info['pattern'] = np.sum(calculate_poisson_probabilities_log(
             cevns_points[st_mask]['pe_by_area'][:,:64], cevns_points[st_mask]['recons_light_pattern'][:,:64]),axis = 1)
    score_cevns = cevns_info['pattern'] - (k_st * cevns_info['st_cor']) + k_area * cevns_info['area'] + b
    
    cevns_pass_pattern = cevns_info[score_cevns > 0]
    
    cevns_id_remain = cevns_id[score_cevns > 0]
    
    cevns_pe = cevns_sim.cevns_pe_info[i]
    
    ########
    # print(cevns_pe['event_id'][-1])
    # print(np.max(cevns_id))
    print(len(cevns_id_remain) / len(cevns_id))
    
    accumulate_num += len(cevns_points)
    
    max_num = np.min([10,len(cevns_id_remain)])
    
    waveform_list = []
    std_list = []
    waveform_list_top = []
    std_list_top = []
    z_list = np.linspace(0,24,max_num)
    for j in tqdm(range(max_num)):
        
        cevns_id_event = cevns_id_remain[j]
        print('cevns_id_event: ', cevns_id_event)

        pe_mask = (cevns_pe['event_id']==cevns_id_event) 
        pe_info = cevns_pe[pe_mask]
        # print(len(pe_info))
        pe_num = len(pe_info)
        pe_area = np.sum(pe_info['area'])
#         print("area1: ", pe_area / 6e6)
        
#         print("area2: ", cevns_pass_pattern[j]['area'])
        
#         print("Score: ", score_cevns[score_cevns > 0][j])

        waveform,std = cevns_waveform_gen(pe_info,sigma_se,z_list[j])
        # waveform_top,std_top = top_waveform_gen(pe_info,sigma_se)
        waveform[waveform <0] = 0
        waveform_list.append(waveform)
        std_list.append(std)
    np.save('z_pass_pattern_cevns_3_' + str(e_num), z_list)
    np.save('wf_pass_pattern_cevns_3_' + str(e_num), waveform_list)
    np.save('wf_std_pass_pattern_cevns_3_' + str(e_num), std_list)
    np.save('area_cevns_3_' + str(e_num), cevns_info['area'])


0.0


0it [00:00, ?it/s]


0.03869


 30%|███████████████████████████████▌                                                                         | 3/10 [00:00<00:00, 26.73it/s]

cevns_id_event:  93
cevns_id_event:  101
cevns_id_event:  108
cevns_id_event:  122
cevns_id_event:  137
cevns_id_event:  150


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 30.69it/s]

cevns_id_event:  152
cevns_id_event:  187
cevns_id_event:  191
cevns_id_event:  192


波形鉴别
=====

In [ ]:
import torch
import torch.nn as nn
import pickle
from PIL import Image
import h5py

#神经网络的结构，得跑一下才能把后面有些定义的变量弄进去，不耗时间
class WaveformClassifier(nn.Module):
    def __init__(self):
        super(WaveformClassifier, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2)
        )
        self.global_pool = nn.AdaptiveAvgPool1d(1)  # 使用全局平均池化层
        self.fc = nn.Sequential(
            nn.Linear(128, 64),  # 修改全连接层的输入维度
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(32, 2)
        )
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.global_pool(x).squeeze(2)
        x = self.fc(x)
        return x
    

In [ ]:
# 指定cevns文件夹路径
folder_path_cevns = 'wf_cevns_3_3.npy'
folder_path_cevns_std = 'wf_std_cevns_3_3.npy'

In [ ]:
test_cevns_data = np.load(folder_path_cevns)
test_cevns_std = np.load(folder_path_cevns_std)

In [ ]:
# 转换数据通道数为1并调整大小
test_cevns_data = test_cevns_data.reshape(-1, 1, 3500)
test_cevns_labels = np.ones(len(test_cevns_data))
test_cevns_pe = np.sum(np.sum(test_cevns_data,axis = 1),axis = 1)

test_data = test_cevns_data
test_labels = test_cevns_labels
test_pe = test_cevns_pe

In [ ]:
#把测试集的名字换成我们需要的，test_filename = 'test3.pkl'，这个。然后用已经做好的模型model.load_state_dict(torch.load('waveform_classifier.pth'))得到一个分数
#这个分数存在h5_filename = 'scores3.h5'这里面

# # 加载测试集数据和标签
# test_filename = 'test3.pkl'
# 长度为3500的波形？
# train_loss_values = []  # 保存训练损失值的列表
# test_loss_values = []   # 保存测试损失值的列表

# with open(test_filename, 'rb') as f:
#     test_data, test_labels, test_pe = pickle.load(f)



# # 转换数据通道数为1并调整大小
# test_data = test_data.reshape(-1, 1, 3500)

# 将测试集数据和标签转换为PyTorch张量
test_data = torch.from_numpy(test_data).float()
test_labels = torch.from_numpy(test_labels).long()

# 创建测试集数据集和数据加载器
test_dataset = torch.utils.data.TensorDataset(test_data, test_labels)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

# 加载模型
model = WaveformClassifier()
model.load_state_dict(torch.load('../waveform_classifier.pth'))


# 测试模式
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0
test_confusion_matrix = torch.zeros(2, 2)
criterion = nn.CrossEntropyLoss()

num_epochs = 2
with torch.no_grad():
    for batch_data, batch_labels in test_dataloader:
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)
        test_loss += loss.item() * batch_data.size(0)

        # 计算测试集准确率和混淆矩阵
        _, predicted = torch.max(outputs, 1)
        test_total += batch_labels.size(0)
        test_correct += (predicted == batch_labels).sum().item()

        for i in range(len(predicted)):
            test_confusion_matrix[batch_labels[i]][predicted[i]] += 1

# 计算平均测试集损失函数和准确率
test_loss /= len(test_dataset)
test_accuracy = test_correct / test_total

# 计算标签0判断为标签1和标签1判断为标签0的概率
test_false_positive = test_confusion_matrix[0][1] / test_confusion_matrix[0].sum()
test_false_negative = test_confusion_matrix[1][0] / test_confusion_matrix[1].sum()

print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")
print(f"Test False Positive Rate: {test_false_positive:.4f}, Test False Negative Rate: {test_false_negative:.4f}")


# 在测试集上进行评估
model.eval()
test_confusion_matrix = torch.zeros(2, 2)

# 在测试集上计算预测分数
with torch.no_grad():
    model.eval()
    outputs = model(test_data)
    scores = torch.softmax(outputs, dim=1)

# 绘制预测分数的分布
scores_label_0 = scores[test_labels == 0]
scores_label_1 = scores[test_labels == 1]


In [ ]:
score = scores_label_1[:, 0].numpy()
mask_wf = (score > 0.8)
mask_width = (test_cevns_std * 1e6 > 0.22)

acceptance_wf_width = len(score[mask_wf * mask_width]) / len(score)
acceptance_wf_width,acceptance_pattern

In [ ]:
test_cevns_data = np.load(folder_path_cevns)
test_cevns_std = np.load(folder_path_cevns_std)
area_range = np.sum(test_cevns_data,axis = 1)
z_range = np.linspace(0,24,len(test_cevns_data))

In [ ]:
cut = np.zeros(len(score))
cut[mask_wf * mask_width] = 1

pe_bins = np.arange(90,250,10)
z_bins = np.arange(0,25,1)
for i in range(len(pe_bins)-1):
    pe_low, pe_up = pe_bins[i], pe_bins[i+1]
    print("pe_low, pe_up: ", pe_low, pe_up)
    pe_mask = (area_range > pe_low) * (area_range < pe_up)
    for j in range(len(z_bins)-1):
        z_low, z_up = z_bins[j], z_bins[j+1]
        print("z_low, z_up: ", z_low, z_up)
        z_mask = (z_range > z_low) * (z_range < z_up)
        mask = pe_mask * z_mask
        print(len(cut[mask]))
        
    

In [ ]:
cut,cevns_info['area']

In [ ]:
cut = np.zeros(len(score))
cut[mask_wf * mask_width] = 1
splits_cut = np.array_split(cut, 10)
splits_score = np.array_split(np.ones(len(score)), 10)
acceptance_list = np.array([np.sum(split) for split in splits_cut])
num_list = np.array([np.sum(split) for split in splits_score])
acceptance = acceptance_list / num_list
np.save("acceptance_3", acceptance)

In [ ]:
z_dis = np.linspace(0,240,10)
plt.figure()
plt.style.use('../relics.mplstyle')
plt.plot(z_dis,acceptance)
plt.xlabel(r'$\mathrm{Z [mm]}$')
plt.ylabel(r'$\mathrm{Acceptance}$')
plt.title(r'$\mathrm{4e\ Event\ Acceptance}$')
plt.grid()
plt.ylim([0, 1])
plt.savefig("3e_accept.png",dpi = 200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
# 创建一个新的图形并设置大小
fig= plt.figure()
plt.hist(scores_label_0[:, 0].numpy(), bins=np.arange(0,1,0.01), edgecolor='black', alpha=0.5, label='DE', density=True, color='darkblue')
plt.hist(scores_label_1[:, 0].numpy(), bins=np.arange(0,1,0.01), edgecolor='black', alpha=0.5, label=r"CE$\nu$NS", density=True, color='darkgreen')

plt.xlabel('Waveform Classifier Score', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
#plt.title('Distribution of Prediction Scores')
plt.legend()
# plt.savefig('defen.svg', transparent=True, dpi=400, bbox_inches='tight')
plt.axvline(x=0.8, color='red', linestyle='dashed')
plt.grid()
plt.show()


# h5_filename = 'scores3.h5'
# h5_file = h5py.File(h5_filename, 'w')

# # 将得分存储为名为'scores'的数据集
# h5_file.create_dataset('scores', data=scores.numpy())

# # 关闭HDF5文件
# h5_file.close()


最终结果
=======

In [ ]:
# de_wf_mask = (scores_label_0[:, 0].numpy() > 0.8)
cevns_wf_mask = (scores_label_1[:, 0].numpy() > 0.8)

len(test_cevns_pe[cevns_wf_mask]) / len(test_cevns_pe)

In [ ]:
bg = np.load('e_spectrum_relics.npz')
cevns_e_spectrum = bg['CEvNS'][3:8] * 30 * 0.8# y-1 with 20ms deat time after each muon
cevns_e_spectrum

In [ ]:
ratio = 1e5/np.sum(cevns_e_spectrum)

In [ ]:
cevns_hist

In [ ]:
cevns_hist_original

In [ ]:
import matplotlib.ticker as ticker

pile_hist, events_bins = np.histogram(test_de_pe[de_wf_mask], bins = np.arange(120,260,10))
cevns_hist,events_bins_original = np.histogram(test_cevns_pe[cevns_wf_mask], bins = np.arange(120,260,10))
pile_hist_original,events_bins_original = np.histogram(test_de_pe, bins = np.arange(120,260,10))
cevns_hist_original,events_bins_original = np.histogram(test_cevns_pe, bins = np.arange(120,260,10))

fig, ax = plt.subplots(figsize=(8, 6), dpi=80)

# 计算每个柱的宽度
width = (events_bins[1:] - events_bins[:-1])

# 计算误差
error = np.sqrt(pile_hist)
error_original = np.sqrt(pile_hist_original)
error_cevns = np.sqrt(cevns_hist)
error_cevns_original = np.sqrt(cevns_hist_original)

# 使用ax.errorbar创建散点图并添加误差条
ax.errorbar((events_bins[1:] + events_bins[:-1])/2, pile_hist/0.09726406367944956/30/10, yerr=error/0.09726406367944956/30/10, fmt='o',color = "skyblue",label = "DE after wf-cut")
ax.errorbar((events_bins[1:] + events_bins[:-1])/2, pile_hist_original/0.09726406367944956/30/10, yerr=error_original/0.09726406367944956/30/10, fmt='o',color = "forestgreen",label = "DE without wf-cut")
ax.errorbar((events_bins[1:] + events_bins[:-1])/2, cevns_hist_original / ratio /30 / 10 * 0.8, yerr=error_cevns_original/ratio/30/10, fmt='o',color = "darkorange",label = "CEvNS")

ax.set_ylabel(r"Rate  $\mathrm{[kg^{-1} year^{-1} pe^{-1}]}$")
ax.set_yscale("log")

# 设置y轴的刻度为每个10的n次方
ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0))

# 增加背景刻度线
ax.grid(which='major', axis='y')

plt.legend()

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker

# fig, ax = plt.subplots(figsize=(8, 6), dpi=80)
fig, ax = plt.subplots()

# 新增变量 bin_centers 存储bins中点位置
bin_centers = (events_bins[1:] + events_bins[:-1])/2

# 使用ax.plot画出类似直方图的线段
ax.plot(bin_centers, pile_hist/0.09726406367944956/30/10, drawstyle='steps-mid',color="forestgreen")
# ax.plot(bin_centers, pile_hist_original/0.09726406367944956/30/10, drawstyle='steps-mid',color="")
ax.plot(bin_centers, cevns_hist / ratio /30 / 10 * 0.8, drawstyle='steps-mid',color="black")

# 使用ax.errorbar只添加误差条，fmt='none'表示不显示原有的点标记
ax.errorbar(bin_centers, pile_hist/0.09726406367944956/30/10, yerr=error/0.09726406367944956/30/10, fmt='none',color = "forestgreen")
# ax.errorbar(bin_centers, pile_hist_original/0.09726406367944956/30/10, yerr=error_original/0.09726406367944956/30/10, fmt='none',color = "")
ax.errorbar(bin_centers, cevns_hist / ratio /30 / 10 * 0.8, yerr=error_cevns_original/ratio/30/10, fmt='none',color = "black")

ax.set_ylabel(r"Rate  $\mathrm{[kg^{-1} year^{-1} pe^{-1}]}$")
ax.set_xlabel(r"S2 [pe]")
ax.set_yscale("log")

# 设置y轴的刻度为每个10的n次方
ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0))

# 增加背景刻度线
ax.grid(which='major', axis='y')

plt.legend()

plt.show()

In [ ]:
bin_centers.shape

In [ ]:
pile_hist.shape

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np

pile_hist, events_bins = np.histogram(test_de_pe[de_wf_mask], bins = np.arange(120,260,10))
cevns_hist,events_bins_original = np.histogram(test_cevns_pe[cevns_wf_mask], bins = np.arange(120,260,10))
pile_hist_original,events_bins_original = np.histogram(test_de_pe, bins = np.arange(120,260,10))
cevns_hist_original,events_bins_original = np.histogram(test_cevns_pe, bins = np.arange(120,260,10))

fig, ax = plt.subplots(figsize=(8, 6), dpi=80)

bin_centers = (events_bins[1:] + events_bins[:-1])/2  # calculate the bin centers

# extend the bins and histograms
bins_extended = np.zeros(len(bin_centers) + 2)
bins_extended[1:-1] = bin_centers
bins_extended[0] = events_bins[0] - (events_bins[1] - events_bins[0])/2
bins_extended[-1] = events_bins[-1] + (events_bins[-1] - events_bins[-2])/2

pile_hist_extended = np.zeros(len(pile_hist) + 2)
pile_hist_extended[1:-1] = pile_hist

pile_hist_original_extended = np.zeros(len(pile_hist_original) + 2)
pile_hist_original_extended[1:-1] = pile_hist_original

cevns_hist_extended = np.zeros(len(cevns_hist) + 2)
cevns_hist_extended[1:-1] = cevns_hist

error = np.sqrt(pile_hist)  # calculate error
error_extended = np.zeros(len(error) + 2)
error_extended[1:-1] = error

error_original = np.sqrt(pile_hist_original)
error_original_extended = np.zeros(len(error_original) + 2)
error_original_extended[1:-1] = error_original

error_cevns = np.sqrt(cevns_hist)
error_cevns_extended = np.zeros(len(error_cevns) + 2)
error_cevns_extended[1:-1] = error_cevns

# plot with step-mid and error bar
ax.plot(bins_extended, pile_hist_extended/0.09726406367944956/30/10, drawstyle='steps-mid',color="forestgreen", label="Delayed Electrons")
ax.errorbar(bin_centers, pile_hist/0.09726406367944956/30/10, yerr=error/0.09726406367944956/30/10, fmt='none',color = "forestgreen")

# ax.plot(bins_extended, pile_hist_original_extended/0.09726406367944956/30/10, drawstyle='steps-mid',color="forestgreen", label="DE without wf-cut")
# ax.errorbar(bin_centers, pile_hist_original/0.09726406367944956/30/10, yerr=error_original/0.09726406367944956/30/10, fmt='none',color = "forestgreen")

ax.plot(bins_extended, cevns_hist_extended / ratio /30 / 10 * 0.8, drawstyle='steps-mid',color = "black", label="CEvNS")
ax.errorbar(bin_centers, cevns_hist_original / ratio /30 / 10 * 0.8, yerr=error_cevns/ratio/30/10, fmt='none',color = "black")

ax.set_ylabel(r"Rate  $\mathrm{[kg^{-1} year^{-1} pe^{-1}]}$",fontsize = 14)
ax.set_yscale("log")
ax.set_xlabel("S2 [PE]",fontsize = 14)

ax.yaxis.set_major_locator(ticker.LogLocator(base=10.0))

ax.grid(which='major', axis='y')

plt.legend(loc='upper right')

plt.show()